In [ ]:
from typing import cast

import numpy as np
np.random.seed(42)
import matplotlib.pyplot as plt

import pandas as pd

from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

In [ ]:
result_df = pd.DataFrame(columns=['Model', 'RMSE_test', 'RMSE_train', 'MAE_test', 'MAE_train'])

In [ ]:
import seaborn as sns
df = sns.load_dataset('diamonds')

train_df, test_df = train_test_split(df, test_size=0.2)

In [ ]:
df[["cut", "color", "clarity"]].nunique()

In [ ]:
def evaluate(model, model_name, train_df, test_df, result_df):
    preds_train = model.predict(train_df.drop('price', axis=1))
    preds_test = model.predict(test_df.drop('price', axis=1))
    
    rmse_train = root_mean_squared_error(train_df['price'], preds_train)
    rmse_test = root_mean_squared_error(test_df['price'], preds_test)
    mae_train = mean_absolute_error(train_df['price'], preds_train)
    mae_test = mean_absolute_error(test_df['price'], preds_test)
    
    result_df.loc[len(result_df)] = [model_name, rmse_test, rmse_train, mae_test, mae_train]

    print(f'Train RMSE: {rmse_train}')
    print(f'Train MAE: {mae_train}')
    print(f'Test RMSE: {rmse_test}')
    print(f'Test MAE: {mae_test}')


In [ ]:
from sklearn.linear_model import LinearRegression

train_non_cat_df = train_df.select_dtypes(exclude='category')
test_non_cat_df = test_df.select_dtypes(exclude='category')

model_lr = LinearRegression()
model_lr.fit(train_non_cat_df.drop('price', axis=1), train_non_cat_df['price'])

evaluate(model_lr, 'Regression non-categorical', train_non_cat_df, test_non_cat_df, result_df)


2. ????????? ??????? ??????? ???????? (????????? ??????????? ????????? ?????????). ???????? ? ???????? ??????????.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [5, 6, 7],
    'n_estimators': [50, 100, 200, 500, 600],
    'learning_rate': [0.01]
}

model = GradientBoostingRegressor(random_state=42)

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
grid.fit(train_non_cat_df.drop('price', axis=1), train_non_cat_df['price'])

In [ ]:
model_gb = GradientBoostingRegressor(random_state=42, **grid.best_params_)
model_gb.fit(train_non_cat_df.drop('price', axis=1), train_non_cat_df['price'])

evaluate(model_gb, 'Boosting non-categorical', train_non_cat_df, test_non_cat_df, result_df)

3. ????????? one hot ? target encoding ??? ?????????????? ????????? (????? ??????, ??? ????? ???????? ????). ???????? ????????.

In [ ]:
train_df_te, test_df_te = train_df.copy(), test_df.copy()

cat_cols = df.select_dtypes(include='category').columns.to_list()

for col in cat_cols:
    mapping = train_df.groupby(col, observed=False)["price"].mean()
    train_df_te[col] = train_df_te[col].map(mapping)
    test_df_te[col] = test_df_te[col].map(mapping)

model_te_reg = LinearRegression()
model_te_reg.fit(train_df_te.drop('price', axis=1), train_df_te['price'])
evaluate(model_te_reg, 'Regression target encoding', train_df_te, test_df_te, result_df)


model_te = GradientBoostingRegressor(random_state=42, **grid.best_params_)
model_te.fit(train_df_te.drop('price', axis=1), train_df_te['price'])
evaluate(model_te, 'Boosting target encoding', train_df_te, test_df_te, result_df)


In [ ]:
train_df_te, test_df_te = train_df.copy(), test_df.copy()

train_ohe_df = pd.get_dummies(train_df, columns=cat_cols, drop_first=True)
test_ohe_df  = pd.get_dummies(test_df,  columns=cat_cols, drop_first=True)

model_one_hot_reg = LinearRegression()
model_one_hot_reg.fit(train_ohe_df.drop('price', axis=1), train_ohe_df['price'])
evaluate(model_one_hot_reg, 'Regression one-hot encoding', train_ohe_df, test_ohe_df, result_df)

model_one_hot = GradientBoostingRegressor(random_state=42, **grid.best_params_)
model_one_hot.fit(train_ohe_df.drop('price', axis=1), train_ohe_df['price'])

evaluate(model_one_hot, 'Boosting one-hot encoding', train_ohe_df, test_ohe_df, result_df)


In [ ]:
result_df.head(6)

4. ??????????????? ??????? (?? ??????????? ??? ??? ?? ???????) ? ???????????? ??????? ?????????.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(train_df_te.select_dtypes(exclude='category').drop(["price"], axis=1))

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.scatter(X_pca[:,0], X_pca[:,1], alpha=0.6, s=1)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA ????????")
plt.grid(True)
plt.show()